## 1. Installation: dtach, opencode, and nexus-lite
Install system dependency `dtach`, the `opencode` CLI (for zero-signup free models), and the `nexus-lite` package directly from GitHub.

> **Note**: Google Colab pre-installs Antigravity CLI (`agy`) since 2026-09-16 alongside official `google-colab-cli` and `colab-mcp`. Combined with `opencode`, multi-agent `agy` + `opencode` swarms are immediately ready on `PATH`.

In [ ]:
!apt-get update -qq && apt-get install -y -qq dtach
!($HOME/.opencode/bin/opencode --version 2>/dev/null | grep -q 'v2\.') || (curl -fsSL https://opencode.ai/v2/install | bash)
!pip install -q --upgrade --force-reinstall --no-deps git+https://github.com/choilab-jefferson/nexus.lite.git && pip install -q pytest 'pyte>=0.8.1'

## 2. Google Drive Mount & Safe Workspace / .gemini Credential Sync
Mount Google Drive for persistent storage across Colab sessions.
`nl.ensure_colab(drive_path=..., gemini_drive_path=...)` safely points the workspace to Google Drive and links `~/.gemini` credentials without overwriting existing files:
- If a file exists in both local and Drive, Drive wins (existing Drive credentials are never overwritten).
- Local-only files are safely preserved into Drive before symlinking.

In [ ]:
# (b) Google Drive mount & safe workspace / .gemini credential sync
try:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_mounted = True
except Exception as e:
    print(f"Not running in live Colab or Drive mount skipped: {e}")
    drive_mounted = False

import shutil
import sys
for _m in [k for k in list(sys.modules) if k == "nexus_lite" or k.startswith("nexus_lite.")]:
    del sys.modules[_m]
import nexus_lite as nl

# Mount workspace, install/locate opencode, and safely sync ~/.gemini credentials
# Existing Drive credentials/files are NEVER overwritten
if drive_mounted:
    ws = nl.ensure_colab(
        drive_path="/content/drive/MyDrive/nexus_workspace",
        gemini_drive_path="/content/drive/MyDrive/.gemini",
        install_opencode=True,
    )
else:
    ws = nl.ensure_colab(install_opencode=True)

print(f"Workspace initialized at: {ws}")
print(f"Detected agy CLI       : {shutil.which('agy')}")
print(f"Detected opencode CLI  : {ws.get('opencode_bin') or shutil.which('opencode')}")

## 3. Cloudflared Remote Colab Bridge & Live Dashboard URL
Start an HTTP JSON bridge server and cloudflared trycloudflare tunnel with `nl.start_bridge()`.
Starting the bridge right at the beginning provides a live dashboard URL and allows external agents (Claude, Agy, Nexus) or browser clients to inspect and drive all subsequent PTY sessions, swarms, and pipelines in real time.

In [ ]:
# (c) Start Cloudflared Remote Colab Bridge & RemoteColab client
# In live Colab, set tunnel=True for a public https://*.trycloudflare.com URL
# A public tunnel requires an explicit token (it is the only auth on the public URL)
import secrets
bridge_token = secrets.token_hex(16)
bridge_info = nl.start_bridge(port=8765, tunnel=True, token=bridge_token)
print("Bridge Info:", {k: v for k, v in bridge_info.items() if k != "token"})  # never print the token
print(f"Local URL: {bridge_info['local_url']}")
print(f"Public URL: {bridge_info.get('public_url')}")
print(f"Live Dashboard URL: {bridge_info['dashboard_url']}")
if bridge_info.get("colab_dashboard_url"):
    print(f"Colab Dashboard URL: {bridge_info['colab_dashboard_url']}")

# Connect via RemoteColab client snippet
remote = nl.RemoteColab(bridge_info["local_url"], token=bridge_token)
print("Remote Health:", remote.health())

# Note: Do NOT call nl.stop_bridge() automatically at the end of the notebook!
# The HTTP bridge server and cloudflared tunnel run in persistent dtach background processes
# so external agents, monitors, and browser dashboards remain connected.
# Calling stop_bridge() terminates the background tunnel, causing immediate 502 Bad Gateway / Cloudflare Error 1033.


## 4. Persistent PTY Sessions & Live Agent-Bus Pointer Injection
Spawn persistent terminal sessions via `dtach`, send input, read terminal screens, and inject live message pointers directly into active PTY sessions. Subtasks can also be delegated with automatic markdown artifact logging and indexed in SQLite FTS5 wiki.

In [ ]:
# (d) Persistent PTY sessions, live message bus pointer injection, and FTS5 wiki
nl.pty.kill("agent-alpha")  # clear a stale session left by an earlier/interrupted run
sess = nl.pty.spawn("bash --norc", name="agent-alpha")
nl.pty.send(sess, "echo 'Agent Alpha online'")
screen = nl.pty.read(sess)
print("Initial PTY Screen:\n", screen)

# Send a message via message bus; pointer will be injected directly into agent-alpha PTY
msg = nl.bus_send(target=sess, message="Coordinate on task #101", sender="orchestrator", kind="dm")
print("Bus Message Sent:", msg["id"])

# Read screen again to observe the live [nexus] pointer injection
screen_after = nl.pty.read(sess)
print("PTY Screen After Bus Send:\n", screen_after)

# Clean up session
nl.pty.kill(sess)

# Sub-agent delegation and FTS5 markdown wiki
res = nl.delegate(
    agent_cmd="python3 -c \"print('Subtask completed successfully!')\"",
    prompt="Compute project metrics",
    caller="main",
    bg=False
)
print("Delegation Result:", res)

nl.wiki_write("architecture/protocols", "# Protocols\nNexus-Lite provides zero-infrastructure MTM and PTY control.")
nl.wiki_write("architecture/database", "# Database\nSQLite WAL mode ensures fast concurrent writes without lock contention.")

search_hits = nl.wiki_search("SQLite WAL")
print("FTS5 Wiki Search Results:", search_hits)

## 5. Multi-Agent Swarm (run_swarm), Pipeline (run_pipeline), and Inline Visualization Dashboard (nl.dashboard())
Execute parallel swarms (`nl.run_swarm`) across multiple workers with automatic synchronization and artifact generation, run sequential multi-stage pipelines (`nl.run_pipeline`: e.g. driver &rarr; verifier &rarr; documenter) with inter-stage handoffs over the agent bus, and inspect the real-time multi-agent visualization dashboard.

> **Tip (Colab Fullscreen Output Sharing & Live Dashboard URL)**: Alongside the Cloudflared Live Dashboard URL from Section 3, Google Colab provides native **Fullscreen Output Sharing**: click the three dots (`⋮`) on the top-right of the dashboard output cell and select **"View output fullscreen"** or **"Copy link to fullscreen output"** ("Share dashboard") to monitor the live multi-agent swarm in a dedicated browser tab!

In [ ]:
# (e) Multi-Agent Swarm (run_swarm), Pipeline (run_pipeline), and Inline Dashboard (nl.dashboard())
swarm_specs = [
    {
        "task_id": "metrics-worker",
        "cmd": "python3 -c \"import json; print(json.dumps({'cpu': 'healthy', 'latency_ms': 12}))\"",
        "prompt": "Compute system health metrics",
    },
    {
        "task_id": "analyzer-worker",
        "cmd": "python3 -c \"print('Analysis complete: cluster operating within normal parameters.')\"",
        "prompt": "Run cluster log analysis",
    },
    {
        "task_id": "indexer-worker",
        "cmd": "python3 -c \"import nexus_lite as nl; nl.wiki_write('reports/cluster', '# Cluster Status\\nAll nodes nominal.'); print('Indexed cluster report to wiki.')\"",
        "prompt": "Index status into wiki",
    },
]

print("Launching 3-worker swarm...")
swarm_results = nl.run_swarm(swarm_specs, caller="orchestrator", timeout=30.0)
for res in swarm_results:
    print(f"[{res['status'].upper()}] {res['task_id']}: {res['output'].strip()}")

# Multi-Agent Sequential Pipeline (driver -> verifier -> documenter)
pipeline_stages = [
    {
        "name": "driver",
        "cmd": "python3 -c \"open('math_tool.py', 'w').write('def square(x): return x * x\\n'); print('Wrote math_tool.py')\"",
        "prompt": "Implement square() function",
    },
    {
        "name": "verifier",
        "cmd": "python3 -c \"import math_tool; assert math_tool.square(9) == 81; print('Square function verified: 9^2 == 81')\"",
        "prompt": "Verify math_tool.py: {prev_output}",
    },
    {
        "name": "documenter",
        "cmd": "python3 -c \"import nexus_lite as nl; nl.wiki_write('modules/math_tool', '# Math Tool\\n\\nsquare(x) verified.'); print('Wiki documentation updated.')\"",
        "prompt": "Document math_tool: {prev_output}",
    },
]

print("\nRunning 3-stage pipeline (driver -> verifier -> documenter)...")
pipeline_res = nl.run_pipeline(pipeline_stages, initial_prompt="Create and verify math tool", caller="orchestrator")
print("Final Pipeline Output:\n", pipeline_res["final_output"])

# Render self-contained HTML5 dashboard inline in Colab output cell
nl.dashboard()

## 6. Live 'agy' Multi-Agent Workflows: Oneshot, Parallel Swarm, Role Pipeline, Interactive PTY & Live Dashboard
Demonstrate real-world multi-agent workflows driving the Antigravity CLI (`agy`) directly in Colab:
- **(a) Oneshot Delegation**: Non-interactive single-shot analysis via `nl.delegate('agy', ...)`.
- **(b) Parallel Swarm**: Concurrent multi-agent execution via `nl.run_swarm` with specialized workers (`agy-architect` and `agy-security`).
- **(c) Role Pipeline**: Sequential multi-stage handoffs via `nl.run_pipeline` with Open-Core roles (`driver` &rarr; `verifier`) and automatic FTS5 wiki persistence.
- **(d) Interactive PTY & Dashboard**: Persistent conversational session via `nl.pty.spawn` and `nl.chat`, rendered live in `nl.dashboard()` before session cleanup.

> **Tip (Colab Fullscreen Output Sharing & Live Dashboard URL)**: In addition to the Cloudflared Live Dashboard URL generated in Section 3, you can use Google Colab's native **Fullscreen Output Sharing** ("Copy link to fullscreen output" / "Share dashboard" in the cell output menu `⋮`) with the "↻ Auto-Poll (Colab Fullscreen)" toggle button to monitor active PTY terminals and swarms in real time!

In [ ]:
# (f) Full Live 'agy' Multi-Agent Demo: Oneshot, Parallel Swarm, Role Pipeline, and Interactive PTY
import shutil
import sys
for _m in [k for k in list(sys.modules) if k == "nexus_lite" or k.startswith("nexus_lite.")]:
    del sys.modules[_m]
import nexus_lite as nl

if shutil.which("agy"):
    # (a) Oneshot 'agy' delegation
    print("=== (a) Oneshot 'agy' Delegation ===")
    oneshot_res = nl.delegate(
        "agy",
        prompt="Reply with ONE sentence explaining why SQLite WAL is ideal for multi-agent state.",
        caller="orchestrator",
        timeout=60.0,
    )
    output = oneshot_res.get("output", "")
    rc = oneshot_res.get("rc", 0)
    print(f"[{oneshot_res.get('status', 'unknown').upper()}] (rc={rc}):\n{output.strip()}")

    # Check for authentication requirement or execution failure
    if rc != 0 or "authentication required" in output.lower():
        print(
            "\n[Tip: agy Authentication] If unauthenticated, run 'agy' once in Colab Terminal to log in,\n"
            "or mount Google Drive in Step 2 to automatically sync your persistent ~/.gemini credentials."
        )

    # (b) Parallel 'agy' Multi-Agent Swarm (agy-architect and agy-security)
    print("\n=== (b) Parallel 'agy' Multi-Agent Swarm ===")
    swarm_specs = [
        {
            "task_id": "agy-architect",
            "cmd": "agy",
            "prompt": "Analyze microservice design: suggest 2 key architectural patterns in 2 bullet points.",
        },
        {
            "task_id": "agy-security",
            "cmd": "agy",
            "prompt": "Analyze microservice design: highlight 2 top security practices in 2 bullet points.",
        },
    ]
    swarm_results = nl.run_swarm(swarm_specs, caller="orchestrator", timeout=90.0)
    for res in swarm_results:
        print(f"[{res['status'].upper()}] {res['task_id']}:\n{res['output'].strip()}\n")

    # (c) Sequential 'agy' Role Pipeline (driver -> verifier) with Open-Core roles & FTS5 wiki save
    print("=== (c) Sequential 'agy' Role Pipeline ===")
    pipeline_stages = [
        {
            "name": "driver",
            "role": "driver",
            "cmd": "agy",
            "prompt": "Draft 2 concise bullet points for exponential backoff: {prev_output}",
        },
        {
            "name": "verifier",
            "role": "verifier",
            "cmd": "agy",
            "prompt": "Verify and refine in 1 sentence: {prev_output}",
        },
    ]
    pipeline_res = nl.run_pipeline(
        pipeline_stages,
        initial_prompt="Design a retry backoff policy in 2 bullet points",
        caller="orchestrator",
        timeout=90.0,
    )
    print("Final Pipeline Output:\n", pipeline_res.get("final_output", "").strip())

    # Save verified result into FTS5-indexed Wiki
    wiki_path = nl.wiki_write("agy/backoff_policy", pipeline_res.get("final_output", ""))
    print(f"\nSaved final output to wiki: {wiki_path}")

    # (d) Interactive 'agy' PTY session + Live Dashboard Render
    print("\n=== (d) Interactive 'agy' PTY Session & Live Dashboard ===")
    sess_name = nl.pty.spawn("agy", name="agy-live")
    try:
        chat_reply = nl.chat(
            "agy-live",
            "Say Pong in one word.",
            provider="agy",
            timeout=45.0,
        )
        print("agy-live PTY reply:", chat_reply.strip())

        # Render dashboard while agy-live is active so screen & all delegations are visible
        nl.dashboard()
    finally:
        nl.pty.kill("agy-live")
        print("Cleaned up agy-live PTY session.")
else:
    print("agy not found; falling back to zero-signup 'opencode' coding agent (opencode/big-pickle)...")
    nl.ensure_opencode(install=True)
    oc_res = nl.run_opencode_free_demo()
    for r in oc_res["swarm"]:
        print(f"[{r['status'].upper()}] {r['task_id']} ({r['agent_cmd']}): {r['output'].strip()}")

## 7. Peer-to-Peer Agent Interaction (`interact`) & Autonomous 5-Phase Problem-Solving Swarm (`cooperate`)

`nexus-lite` supports genuine **Agent-to-Agent (A2A) peer communication** over `[nexus] dm` pointers and `<nexus_stm>` (`stm/1`) shared short-term memory across CLI coding agents (`agy`, `opencode`, and `claude`):

1. **5-Turn Peer-to-Peer Interaction (`nl.run_agent_interaction`)**:
   - `agy-architect` (`agy`) writes a formal specification and sends a `[nexus] dm` pointer to `code-driver`.
   - `code-driver` (`opencode/mimo-v2.6-flash-free` / `agy`) reads the DM and writes the initial implementation.
   - `opencode-reviewer` (`opencode/mimo-v2.6-flash-free`) inspects the draft and sends an adversarial critique DM back to `code-driver`.
   - `code-driver` reloads `<nexus_stm>` (`stm/1`), patches the module on disk, and notifies `agy-architect` for final `VERDICT: PASS` verification.
2. **Autonomous 5-Phase Multi-Agent Problem-Solving Swarm (`nl.run_cooperative_problem_solver`)**:
   - Takes **any arbitrary problem `goal`** and coordinates 4 specialist agents (`thinker-architect`, `algo-driver`, `test-engineer`, `leaf-scout`) plus `peer-verifier` across:
     - **Phase 1**: Contract decomposition into FTS5 Wiki (`coop/architecture_spec`), `stm/1` decision node, and `[nexus] dm` pointer dispatch.
     - **Phase 2**: Parallel `run_swarm` execution (`algo-driver` via `opencode`, `test-engineer` authoring an adversarial `pytest` suite via `agy`, and `leaf-scout` extracting numeric/type invariants via `opencode/big-pickle`).
     - **Phase 3**: `peer-verifier` executes real `pytest` and DMs root-cause failure diagnostics to `algo-driver`.
     - **Phase 4**: Closed-loop `<nexus_stm>` repair rounds (`algo-driver` -> `agy` escalated refiner) until `pytest` passes (`rc=0`).
     - **Phase 5**: Multi-agent consensus `VERDICT: PASS`, `COOP_SOLVE_COMPLETE` broadcast, FTS5 Wiki report, and interactive HTML5 topology dashboard.

In [ ]:
import shutil
import nexus_lite as nl

# Uses live `agy` + `opencode` swarm when `agy` CLI is on PATH, or deterministic Python fallback in offline CI
arch_agent = "agy" if shutil.which("agy") else "python"

coop_res = nl.run_cooperative_problem_solver(
    "Design and implement a 3D Radiomics Z-Score Harmonizer and IQR Outlier Quarantine module in scanner_harmonizer.py "
    "with functions: validate_feature_vector(values), quarantine_iqr_outliers(values, k=1.5), and "
    "harmonize_scanner_batch(records, feature_key, target_mean=0.0, target_std=1.0).",
    module_name="scanner_harmonizer",
    architect_agent=arch_agent,
    verifier_agent="opencode/mimo-v2.6-flash-free",
    max_rounds=2,
    wiki_slug="coop/scanner_harmonizer_live",
)

print("=== Autonomous 5-Phase Multi-Agent Swarm (agy + opencode) ===")
print("Project ID (stm/1):", coop_res["project_id"])
print("Module Written:    ", coop_res["module_path"])
print("Test Suite Written:", coop_res["test_path"])
print(f"Phase-3 Draft Pytest RC: {coop_res['draft_pytest_rc']} -> Phase-5 Final Pytest RC: {coop_res['final_pytest_rc']}")
print("Repair Rounds:", [
    {"round": r["round"], "agent": r["agent"], "pytest_rc": r["pytest_rc"]}
    for r in coop_res["repair_rounds"]
])
print(f"DM Chain ({len(coop_res['dm_chain'])} peer messages):")
for idx, dm in enumerate(coop_res["dm_chain"], 1):
    print(f"  {idx}. {dm['pointer']}")
print("\n=== Final Pytest Verification Output ===")
print(coop_res["final_pytest_output"])
print("\nConsensus Verdict:\n" + coop_res["consensus_verdict"])


## 8. Running Full E2E Test Suite Inside Colab
Clone the repository and execute the full `pytest` verification suite directly inside the Colab runtime.

In [ ]:
!rm -rf /tmp/nexus-lite-src && git clone -q https://github.com/choilab-jefferson/nexus.lite.git /tmp/nexus-lite-src && pytest -v /tmp/nexus-lite-src/tests/


## 9. Optional: loom adapter
Probe for the `loom` agent harness and run a prompt if it is installed (install path TBD).

In [ ]:
from nexus_lite.loom import probe, LoomAdapter, LOOM_INSTALL_URL
pr = probe()
print('loom available:', pr.available, pr.path or pr.error)
if pr.available:
    r = LoomAdapter().run_prompt('Say hello', provider='mock')
    print('status:', r.status)
else:
    print('loom is not installed; install URL is TBD:', LOOM_INSTALL_URL)
